# Jahres-Wärmebedarf → Spitzen-Wärmeleistung
**Mit BDEW-Standardlastprofilen (demandlib)**

Die Geothermie-Kennlinie (`04_nrw_geothermie_kennlinie.ipynb`) arbeitet mit
**Spitzen-Wärmeleistung (kW)**. In der Arbeitsdatei steht aber nur der
**jährliche Wärmebedarf (kWh/Jahr)**.

Dieses Notebook wandelt den Jahresbedarf über BDEW-Wärmelastprofile in die
Spitzenleistung um — dieselbe Methode, die das SESMG verwendet.

Vorgehen:
1. Normiertes BDEW-Wärmeprofil erzeugen (Jahressumme = 1)
2. Den maximalen Stundenwert im Profil bestimmen — das ist die Spitzen-Fraktion
3. Den **Cluster-Repräsentanten** des Wärmebedarfs mit der Spitzen-Fraktion multiplizieren → Spitzenleistung in kW

**Zwei Gebäudeklassen:**
- SFH/TH → `shlp_type = 'EFH'` (Einfamilienhaus)
- MFH/AB → `shlp_type = 'MFH'` (Mehrfamilienhaus)

**Gebäudealtersklasse:** die häufigste — wir prüfen die Spalte
`construction_year` in der Arbeitsdatei, um zu bestätigen, welche Klasse dominiert.

**Warum Cluster-Repräsentanten und nicht der rohe Bedarf?** Die Spitzenleistung
fließt in die Erdsonden-Dimensionierung
(`05_nrw_rasenflaeche_geothermie.ipynb`) und die finalen Archetyp-Kombinationen
(`07_nrw_finaler_datensatz.ipynb`). Beide arbeiten bereits mit Cluster-
Archetypen, nicht mit einzelnen Gebäuden — die Spitzenleistung muss also aus
demselben Archetyp (`heat_cluster_1/2/3` aus `02_nrw_clustering.ipynb`)
abgeleitet werden, nicht aus dem exakten individuellen Bedarf. Dieses Notebook
läuft deshalb **nach** `02_nrw_clustering.ipynb`.

## 1. Konfiguration

In [17]:
import os, time, warnings
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import demandlib.bdew as bdew
import requests

warnings.filterwarnings("ignore")

BASE_DIR      = "."
FILE_GEBAEUDE = os.path.join(BASE_DIR, "nrw_buildings.parquet")
OUT_DIR       = os.path.join(BASE_DIR, "nrw_results")
os.makedirs(OUT_DIR, exist_ok=True)

# -- Wetterstandorte ---------------------------------------------------------
# Dieselben zwei Referenzstaedte wie bei der PV-Berechnung (Abschnitt 9 in 01)
LOCATIONS = {
    "Essen":          {"lat": 51.4028, "lon": 6.9675},
    "Bad Marienberg": {"lat": 50.6497, "lon": 7.9328},
}

# -- Gebäudealtersklassen-Lookup (BDEW) ---------------------------------------
# 7 Altersklassen aus der demandlib
BUILDING_AGE_CLASS_LOOKUP = {
    2:  (0,    1918),
    3:  (1919, 1948),
    5:  (1949, 1978),   # <- in NRW erwartet haeufigste Klasse
    7:  (1979, 1990),
    9:  (1991, 2000),
    10: (2001, 2008),
    11: (2009, 9999)
}

# Windklasse — 1 = windgeschuetzt, typisch urbanes NRW
WIND_CLASS = 1

print("Imports OK")
print(f'Gebaeudealtersklassen: {list(BUILDING_AGE_CLASS_LOOKUP.keys())}')

Imports OK
Gebaeudealtersklassen: [2, 3, 5, 7, 9, 10, 11]


## 2. Häufigste Gebäudealtersklasse im NRW-Datensatz bestimmen

Wir verwenden die häufigste Altersklasse. Dafür prüfen wir die Spalte
`construction_year` und zählen, welche BDEW-Klasse die meisten Gebäude enthält.

In [18]:
gebaeude = gpd.read_parquet(FILE_GEBAEUDE)
print(f'Gebaeude geladen: {len(gebaeude):,}')
jahre = gebaeude["construction_year"]
print(f'\nBaujahr-Spanne: {jahre.min()} — {jahre.max()}')
print(f'Fehlende Baujahre: {jahre.isna().sum():,}')

# BDEW-Altersklasse je Gebaeude zuweisen
def get_age_class(year):
    if pd.isna(year):
        return 5   # Standard: die vermutlich haeufigste Klasse
    for cls, (y_min, y_max) in BUILDING_AGE_CLASS_LOOKUP.items():
        if y_min <= year <= y_max:
            return cls
    return 5

gebaeude["building_age_class"] = jahre.apply(get_age_class)

print("\nGebaeudezahl je Altersklasse:")
age_counts = gebaeude["building_age_class"].value_counts().sort_index()
for cls, count in age_counts.items():
    yr_range = BUILDING_AGE_CLASS_LOOKUP[cls]
    pct = count / len(gebaeude) * 100
    marker = " <- HAEUFIGSTE" if count == age_counts.max() else ""
    print(f'  Klasse {cls:2d}  ({yr_range[0]}-{yr_range[1]}): {count:>10,}  ({pct:.1f}%){marker}')

most_common_class = age_counts.idxmax()
print(f'\nHaeufigste Gebaeudealtersklasse: {most_common_class}')
print(f'Baujahr-Spanne: {BUILDING_AGE_CLASS_LOOKUP[most_common_class]}')

Gebaeude geladen: 4,134,282

Baujahr-Spanne: 1859 — 2022
Fehlende Baujahre: 0

Gebaeudezahl je Altersklasse:
  Klasse  2  (0-1918):    169,351  (4.1%)
  Klasse  3  (1919-1948):    147,718  (3.6%)
  Klasse  5  (1949-1978):  3,463,330  (83.8%) <- HAEUFIGSTE
  Klasse  7  (1979-1990):     69,808  (1.7%)
  Klasse  9  (1991-2000):    138,732  (3.4%)
  Klasse 10  (2001-2008):    128,739  (3.1%)
  Klasse 11  (2009-9999):     16,604  (0.4%)

Haeufigste Gebaeudealtersklasse: 5
Baujahr-Spanne: (1949, 1978)


## 3. BDEW-Wärmeprofile erzeugen

Normierte Wärmelastprofile für beide Standorte und beide Gebäudetypen. Das
Profil ist auf Jahressumme = 1 normiert, daher gibt der Maximalwert direkt den
Anteil des Jahresbedarfs an, der in die Spitzenstunde fällt.

Verwendet wird die in Abschnitt 2 gefundene häufigste Altersklasse.

Die Außentemperaturdaten stammen von der Open-Meteo-Archiv-API
(https://open-meteo.com/en/docs/archive-api), Stundenwerte für 2018.

Beim ersten erfolgreichen Abruf werden die Temperaturdaten in
`wetterdaten_2018.csv` im Projektstamm gesichert — jeder weitere Lauf (und
jeder andere Rechner) liest aus dieser Datei, ohne die API erneut aufzurufen.
Schlägt der Abruf fehl, bricht die Zelle mit einer klaren Meldung ab (eine
Temperaturzeitreihe lässt sich nicht als Still-Ersatz erfinden).

In [19]:
YEAR = 2018
CACHE_DATEI = os.path.join(BASE_DIR, f"wetterdaten_{YEAR}.csv")
year_index = pd.date_range("2018-01-01", periods=8760, freq="h")
temperatures = {}

def temp_spalte(name):
    return f"temp_{name.replace(' ', '_')}"

if os.path.exists(CACHE_DATEI):
    # Cache vorhanden — ganz ohne API-Aufruf laden
    wetter = pd.read_csv(CACHE_DATEI)
    for loc_name in LOCATIONS:
        temperatures[loc_name] = wetter[temp_spalte(loc_name)].astype(float)
    print(f"Wetterdaten aus Cache geladen: {CACHE_DATEI}")
else:
    API_MAX_VERSUCHE = 3
    API_TIMEOUT_S = 30   # ohne timeout haengt der Aufruf minutenlang im ConnectTimeout

    for loc_name, loc in LOCATIONS.items():
        temp_params = {
            "latitude": loc["lat"],
            "longitude": loc["lon"],
            "start_date": f"{YEAR}-01-01",
            "end_date": f"{YEAR}-12-31",
            "hourly": ["temperature_2m"],   # Aussentemperatur
        }
        werte = None
        for versuch in range(1, API_MAX_VERSUCHE + 1):
            try:
                r = requests.get("https://archive-api.open-meteo.com/v1/archive",
                                params=temp_params, timeout=API_TIMEOUT_S)
                r.raise_for_status()
                werte = r.json()["hourly"]["temperature_2m"]
                break
            except requests.RequestException as fehler:
                print(f"  {loc_name}: Versuch {versuch}/{API_MAX_VERSUCHE} fehlgeschlagen "
                      f"({type(fehler).__name__})")
                if versuch < API_MAX_VERSUCHE:
                    time.sleep(5)
        if werte is None:
            raise RuntimeError(
                f"Wetterdaten fuer {loc_name} nicht abrufbar: Die Open-Meteo-API "
                "ist aus diesem Netz nicht erreichbar (Timeout). Netzwerk/Firewall "
                f"pruefen und diese Zelle erneut ausfuehren, oder wetterdaten_{YEAR}.csv "
                "einmalig von einem erreichbaren Netz aus erzeugen und hier ablegen."
            )
        temperatures[loc_name] = pd.Series(werte, dtype=float)
        print(f"Wetterdaten fuer {loc_name} erfolgreich abgerufen.")
        time.sleep(1)

    # Erfolgreichen Abruf als Cache sichern — jeder Folge-Lauf liest aus der Datei
    wetter = pd.DataFrame({"time": year_index})
    for loc_name, temp in temperatures.items():
        wetter[temp_spalte(loc_name)] = temp.values
    wetter.to_csv(CACHE_DATEI, index=False)
    print(f"Als Cache gespeichert: {CACHE_DATEI}")

print("\nTemperaturprofile:")
for loc, temp in temperatures.items():
    print(f'  {loc:<20}: min={temp.min():.1f}°C  mittel={temp.mean():.1f}°C  max={temp.max():.1f}°C')

Wetterdaten aus Cache geladen: ./wetterdaten_2018.csv

Temperaturprofile:
  Essen               : min=-8.1°C  mittel=11.8°C  max=35.5°C
  Bad Marienberg      : min=-13.2°C  mittel=9.4°C  max=31.7°C


In [20]:
# BDEW-Waermeprofile erzeugen
# shlp_type: EFH = Einfamilienhaus, MFH = Mehrfamilienhaus
SHLP_TYPES = {"EFH": ["SFH", "TH"], "MFH": ["MFH", "AB"]}

# Haeufigste Altersklasse aus Abschnitt 2
AGE_CLASS = most_common_class
print(f'Verwendete Gebaeudealtersklasse: {AGE_CLASS}  '
      f'({BUILDING_AGE_CLASS_LOOKUP[AGE_CLASS][0]}-{BUILDING_AGE_CLASS_LOOKUP[AGE_CLASS][1]})')
print()

heat_profiles = {}
peak_fractions = {}

for location, temp in temperatures.items():
    heat_profiles[location] = {}
    peak_fractions[location] = {}
    for shlp_type in SHLP_TYPES.keys():
        profile = bdew.HeatBuilding(
            df_index=year_index,
            temperature=temp,
            shlp_type=shlp_type,
            wind_class=WIND_CLASS,
            annual_heat_demand=1,   # auf 1 normiert
            name=f"{location}_{shlp_type}",
            building_class=AGE_CLASS
        ).get_bdew_profile()

        heat_profiles[location][shlp_type] = profile
        peak_val = profile.max()
        peak_fractions[location][shlp_type] = peak_val

        print(f'  {location:<20} {shlp_type}: Spitzen-Fraktion = {peak_val:.6f}  '
              f'(= {peak_val*1000:.2f} Wh je Wh Jahresbedarf)')

Verwendete Gebaeudealtersklasse: 5  (1949-1978)

  Essen                EFH: Spitzen-Fraktion = 0.000530  (= 0.53 Wh je Wh Jahresbedarf)
  Essen                MFH: Spitzen-Fraktion = 0.000409  (= 0.41 Wh je Wh Jahresbedarf)
  Bad Marienberg       EFH: Spitzen-Fraktion = 0.000462  (= 0.46 Wh je Wh Jahresbedarf)
  Bad Marienberg       MFH: Spitzen-Fraktion = 0.000358  (= 0.36 Wh je Wh Jahresbedarf)


## 4. Wärmeprofile visualisieren

In [21]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

for row, location in enumerate(LOCATIONS):
    for col, shlp_type in enumerate(SHLP_TYPES.keys()):
        ax = axes[row][col]
        profile = heat_profiles[location][shlp_type]
        ax.plot(profile.values, lw=0.5, color="tomato", alpha=0.8)
        ax.axhline(y=profile.max(), color="red", ls="--", lw=1.5,
                   label=f"Spitze = {profile.max():.5f}")
        ax.set_title(f"{location} — {shlp_type}  |  Altersklasse {AGE_CLASS}",
                     fontsize=10, fontweight="bold")
        ax.set_xlabel("Stunde des Jahres", fontsize=9)
        ax.set_ylabel("Normierter Waermebedarf", fontsize=9)
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)

plt.suptitle("BDEW-normierte Waermelastprofile (Jahressumme = 1)",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "bdew_waermeprofile.pdf"), bbox_inches="tight")
plt.close()
print("Profile gespeichert")

Profile gespeichert


## 5. Cluster-Repräsentanten des Wärmebedarfs in Spitzenleistung umwandeln

**Formel:**
```
peak_power_kw = (cluster_waerme_kwh × peak_fraction) × 1000 / 1000
```

`peak_fraction` ist der maximale Stundenwert des normierten Profils. Da das
Profil auf Jahressumme = 1 normiert ist, gibt die Spitzen-Fraktion den Anteil
der Jahresenergie in der ungünstigsten Einzelstunde an. Multiplikation des
Cluster-Repräsentanten (kWh) mit dieser Fraktion ergibt die Spitzenstunden-
Energie (kWh). Da der Zeitschritt 1 Stunde ist, gilt: Spitzenenergie (kWh) =
Spitzenleistung (kW).

Die Umrechnung basiert auf `heat_cluster_1/2/3` (aus der gemeinsamen
Arbeitsdatei, von `02_nrw_clustering.ipynb`), nicht auf dem rohen
`heat_demand_1/2/3` — siehe Notebook-Einleitung.

In [22]:
# shlp_type je Groessenklasse
gebaeude["shlp_type"] = gebaeude["size_class"].map(
    {"SFH": "EFH", "TH": "EFH", "MFH": "MFH", "AB": "MFH"})

# Spitzen-Fraktion je Gebaeude anhand nearest_city + shlp_type
gebaeude["peak_fraction"] = gebaeude.apply(
    lambda r: peak_fractions.get(r["nearest_city"],
              peak_fractions["Essen"])[r["shlp_type"]], axis=1
)

# Spitzenleistung in kW je Sanierungsszenario:
# heat_cluster_X (kWh, Cluster-Repraesentant) x peak_fraction
# = Spitzenstunden-Energie in kWh = Spitzenleistung in kW
gebaeude["peak_heat_power_kw_1"] = gebaeude["heat_cluster_1"] * gebaeude["peak_fraction"]
gebaeude["peak_heat_power_kw_2"] = gebaeude["heat_cluster_2"] * gebaeude["peak_fraction"]
gebaeude["peak_heat_power_kw_3"] = gebaeude["heat_cluster_3"] * gebaeude["peak_fraction"]

print("Beispielwerte:")
print(gebaeude[["id", "size_class", "nearest_city", "shlp_type", "peak_fraction",
               "peak_heat_power_kw_1", "peak_heat_power_kw_2",
               "peak_heat_power_kw_3"]].head().to_string(index=False))

Beispielwerte:
                  id size_class nearest_city shlp_type  peak_fraction  peak_heat_power_kw_1  peak_heat_power_kw_2  peak_heat_power_kw_3
DEA_DENW01AL1000CrNT         TH        Essen       EFH       0.000530              4.224326              2.767425              1.335199
DEA_DENW01AL10005Mtw         TH        Essen       EFH       0.000530              6.489572              2.767425              1.335199
DEA_DENW01AL10005MU2        SFH        Essen       EFH       0.000530              8.320823              8.320823              2.684510
DEA_DENW01AL10005MU4        MFH        Essen       MFH       0.000409             17.219219             11.702902              4.961451
DEA_DENW01AL10005Mu6        MFH        Essen       MFH       0.000409             17.219219             11.702902              2.891280


## 6. In die gemeinsame Arbeitsdatei schreiben

Die Spitzenleistungs-Spalten werden direkt in `nrw_buildings.parquet`
zurückgeschrieben (wie im NL-Repositorium: eine Arbeitsdatei, die jedes
Notebook anreichert).

In [23]:
neue_spalten = ["building_age_class", "shlp_type", "peak_fraction",
                "peak_heat_power_kw_1", "peak_heat_power_kw_2", "peak_heat_power_kw_3"]
gebaeude[neue_spalten] = gebaeude[neue_spalten].round(4)

gebaeude.to_parquet(FILE_GEBAEUDE, index=False)
print(f"ueberschrieben: {FILE_GEBAEUDE}")
print(f'Zeilen: {len(gebaeude):,}  Spalten: {len(gebaeude.columns)}')

ueberschrieben: ./nrw_buildings.parquet
Zeilen: 4,134,282  Spalten: 42
